# 📖 Notebook 4: Schema Evolution

Your schema will change. Features get added, requirements shift, and data grows. The question isn't *if* your schema will evolve — it's *how* you'll handle it without breaking your application.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to add columns, rename fields, and change types safely
- The difference between backward-compatible and breaking changes
- How to run migrations without downtime
- Common schema evolution patterns and pitfalls

## 🛠️ Setup

```bash
cd core-concepts/data-modeling
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "data_modeling_demo",
    "user": "demo",
    "password": "demo"
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def query(sql, params=None):
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cursor.execute(sql, params)
    rows = cursor.fetchall()
    conn.close()
    return rows

def execute(sql, params=None):
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute(sql, params)
    conn.commit()
    conn.close()

def show_columns(table):
    """Show column names and types for a table."""
    cols = query("""
        SELECT column_name, data_type, is_nullable, column_default
        FROM information_schema.columns
        WHERE table_name = %s AND table_schema = 'public'
        ORDER BY ordinal_position;
    """, (table,))
    print(f"📋 Columns in '{table}':")
    print(f"  {'Column':<20} {'Type':<25} {'Nullable':<10} {'Default'}")
    print("  " + "-" * 70)
    for c in cols:
        print(f"  {c['column_name']:<20} {c['data_type']:<25} {c['is_nullable']:<10} {c['column_default'] or ''}")

try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ Connection failed: {e}")

## 🟢 Safe Changes (Backward Compatible)

These changes **won't break** your existing application code:

| Change | Why It's Safe |
|--------|---------------|
| Add a new column with DEFAULT | Existing rows get the default value |
| Add a new nullable column | Existing rows get NULL (code ignores it) |
| Add a new index | Only affects performance, not behavior |
| Add a new table | Existing code doesn't know about it |
| Widen a column (VARCHAR(50) → VARCHAR(100)) | Existing data still fits |

Let's try each one:

In [ ]:
# Before: look at the current users table
show_columns('users')

In [ ]:
# Evolution 1: Add a nullable column (safe — existing rows get NULL)
print("🟢 Migration 1: Add 'avatar_url' column (nullable)")
print("=" * 55)

execute("ALTER TABLE users ADD COLUMN IF NOT EXISTS avatar_url TEXT")

# Verify: existing users have NULL for the new column
sample = query("SELECT id, username, avatar_url FROM users LIMIT 3")
for row in sample:
    print(f"  User {row['id']}: avatar_url = {row['avatar_url']}")

print()
print("✅ Existing rows are unaffected — they just have NULL for the new column.")
print("   Your app can start writing avatar URLs for new users immediately.")

In [ ]:
# Evolution 2: Add a column with a default value
print("🟢 Migration 2: Add 'is_verified' column with DEFAULT false")
print("=" * 55)

execute("ALTER TABLE users ADD COLUMN IF NOT EXISTS is_verified BOOLEAN DEFAULT false")

sample = query("SELECT id, username, is_verified FROM users LIMIT 3")
for row in sample:
    print(f"  User {row['id']}: is_verified = {row['is_verified']}")

print()
print("✅ All existing users default to is_verified = false.")
print("   No data loss, no broken queries.")

In [ ]:
# Evolution 3: Add a new index
print("🟢 Migration 3: Add index on users.created_at")
print("=" * 55)

# CONCURRENTLY avoids locking the table during index creation
# (In a real migration, always use CONCURRENTLY for large tables)
conn = get_db()
conn.autocommit = True  # CREATE INDEX CONCURRENTLY can't run in a transaction
cursor = conn.cursor()
try:
    cursor.execute("CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_users_created ON users(created_at)")
    print("✅ Index created without locking the table.")
except Exception as e:
    print(f"Index already exists or error: {e}")
conn.close()

print()
print("💡 CREATE INDEX CONCURRENTLY is critical for production databases.")
print("   A regular CREATE INDEX locks the entire table during creation.")
print("   CONCURRENTLY takes longer but doesn't block reads or writes.")

In [ ]:
# Show the updated schema
show_columns('users')

## 🔴 Dangerous Changes (Breaking)

These changes **can break** your application or lose data:

| Change | Why It's Dangerous |
|--------|--------------------|
| Drop a column | Code reading that column will crash |
| Rename a column | Same as drop + add — code references the old name |
| Change a column type (narrow) | Existing data may not fit (VARCHAR(100) → VARCHAR(50)) |
| Add NOT NULL without default | Existing NULL rows violate the constraint |
| Drop a table | Everything that reads it breaks |

Let's see what happens and how to handle them safely:

In [ ]:
# Dangerous: Adding NOT NULL without a default
print("🔴 Dangerous: Add NOT NULL column without default")
print("=" * 55)

try:
    execute("ALTER TABLE users ADD COLUMN phone VARCHAR(20) NOT NULL")
    print("✅ Succeeded (unexpected!)")
except Exception as e:
    error_msg = str(e).split('\n')[0]
    print(f"  ❌ Failed: {error_msg}")
    print()
    print("  The database won't let you add a NOT NULL column when existing")
    print("  rows would have NULL values. You'd need to either:")
    print("    1. Add the column as nullable first, backfill data, then set NOT NULL")
    print("    2. Add with a default: ADD COLUMN phone VARCHAR(20) NOT NULL DEFAULT ''")

In [ ]:
# Safe pattern for adding NOT NULL: the 3-step migration
print("🟢 Safe Pattern: Adding a NOT NULL column in 3 steps")
print("=" * 55)

# Step 1: Add as nullable
execute("ALTER TABLE users ADD COLUMN IF NOT EXISTS phone VARCHAR(20)")
print("  Step 1: ALTER TABLE users ADD COLUMN phone VARCHAR(20);")
print("          → Column added as nullable. Existing rows get NULL.")

# Step 2: Backfill existing rows
execute("UPDATE users SET phone = 'unknown' WHERE phone IS NULL")
null_count = query("SELECT COUNT(*) as n FROM users WHERE phone IS NULL")[0]['n']
print(f"  Step 2: UPDATE users SET phone = 'unknown' WHERE phone IS NULL;")
print(f"          → Backfilled. Remaining NULLs: {null_count}")

# Step 3: Now it's safe to add the NOT NULL constraint
execute("ALTER TABLE users ALTER COLUMN phone SET NOT NULL")
print(f"  Step 3: ALTER TABLE users ALTER COLUMN phone SET NOT NULL;")
print(f"          → Constraint added. ✅")

print()
print("💡 This 3-step pattern (add nullable → backfill → set NOT NULL)")
print("   is the standard approach for zero-downtime migrations.")

## 🔄 Renaming a Column Safely

Renaming a column is one of the trickiest migrations because your old application code references the old name and your new code references the new name.

**The safe pattern is a 4-step process:**

1. **Add** the new column
2. **Write to both** old and new columns (dual-write)
3. **Migrate** old data to the new column
4. **Drop** the old column (only after all code is updated)

In [ ]:
# Example: rename 'display_name' to 'full_name'
print("🔄 Safe Rename: 'display_name' → 'full_name'")
print("=" * 55)

# Step 1: Add the new column
execute("ALTER TABLE users ADD COLUMN IF NOT EXISTS full_name VARCHAR(100)")
print("  Step 1: Add 'full_name' column")

# Step 2: Copy data from old to new
execute("UPDATE users SET full_name = display_name WHERE full_name IS NULL")
print("  Step 2: Copy display_name → full_name for all rows")

# Verify both columns have data
sample = query("""
    SELECT id, display_name, full_name
    FROM users LIMIT 3
""")
print()
for row in sample:
    print(f"  User {row['id']}: display_name='{row['display_name']}', full_name='{row['full_name']}'")

print()
print("  Step 3: Deploy new code that reads from 'full_name'")
print("          (Both columns exist during transition)")
print()
print("  Step 4: DROP COLUMN display_name")
print("          (Only after ALL code uses 'full_name')")
print()
print("💡 We won't drop 'display_name' here — in production you'd wait")
print("   days/weeks to ensure no code still references it.")

## 📐 Splitting and Merging Tables

As your system grows, you may need to:
- **Split** a table when it has too many columns or mixed concerns
- **Merge** tables when you realize two entities are really one

In [ ]:
# Splitting: extract user_settings from users table
# This is common when users table grows too wide

print("📐 Table Split: Extract 'user_settings' from 'users'")
print("=" * 55)
print()
print("Before: users table has profile AND settings mixed together.")
print("After:  users table for profile, user_settings for preferences.")
print()

# Create the new table
execute("""
    CREATE TABLE IF NOT EXISTS user_settings (
        user_id INTEGER PRIMARY KEY REFERENCES users(id),
        theme VARCHAR(20) DEFAULT 'light',
        notifications_enabled BOOLEAN DEFAULT true,
        language VARCHAR(10) DEFAULT 'en',
        timezone VARCHAR(50) DEFAULT 'UTC'
    );
""")

# Populate with defaults for existing users
execute("""
    INSERT INTO user_settings (user_id)
    SELECT id FROM users
    ON CONFLICT DO NOTHING;
""")

# Show the split
result = query("""
    SELECT u.username, u.email,
           s.theme, s.notifications_enabled, s.language
    FROM users u
    JOIN user_settings s ON s.user_id = u.id
    LIMIT 3;
""")

for row in result:
    print(f"  @{row['username']}: theme={row['theme']}, "
          f"notifications={row['notifications_enabled']}, lang={row['language']}")

print()
print("💡 Table splitting keeps each table focused on one concern.")
print("   Users rarely change settings, so separating them reduces row size")
print("   for the frequently-read users table.")

## 📝 Migration Tracking

In production, you need to track which migrations have been applied. Here's a simple pattern:

In [ ]:
# A simple migration tracking system

execute("""
    CREATE TABLE IF NOT EXISTS schema_migrations (
        version INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        applied_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
""")

def run_migration(version, name, sql):
    """Run a migration if it hasn't been applied yet."""
    already_applied = query(
        "SELECT 1 FROM schema_migrations WHERE version = %s", (version,)
    )
    if already_applied:
        print(f"  ⏭️  Migration {version} ({name}) — already applied")
        return
    
    execute(sql)
    execute(
        "INSERT INTO schema_migrations (version, name) VALUES (%s, %s)",
        (version, name)
    )
    print(f"  ✅ Migration {version} ({name}) — applied")

print("📝 Running Migrations:")
print("=" * 55)

run_migration(1, "add_avatar_url",
    "ALTER TABLE users ADD COLUMN IF NOT EXISTS avatar_url TEXT")

run_migration(2, "add_is_verified",
    "ALTER TABLE users ADD COLUMN IF NOT EXISTS is_verified BOOLEAN DEFAULT false")

run_migration(3, "add_posts_edit_history",
    "ALTER TABLE posts ADD COLUMN IF NOT EXISTS edited_at TIMESTAMP")

run_migration(4, "add_posts_is_pinned",
    "ALTER TABLE posts ADD COLUMN IF NOT EXISTS is_pinned BOOLEAN DEFAULT false")

print()

# Show migration history
history = query("SELECT * FROM schema_migrations ORDER BY version")
print("📜 Migration History:")
print(f"  {'Version':<10} {'Name':<30} {'Applied At'}")
print("  " + "-" * 65)
for h in history:
    print(f"  {h['version']:<10} {h['name']:<30} {h['applied_at']}")

print()
print("💡 Run the cell again — migrations are idempotent (won't re-apply).")
print("   Tools like Alembic (Python), Flyway (Java), or Knex (Node)")
print("   do this automatically in production.")

## 🧹 Cleanup

In [ ]:
# Clean up the columns and tables we added in this notebook
cleanup_sqls = [
    "ALTER TABLE users DROP COLUMN IF EXISTS avatar_url",
    "ALTER TABLE users DROP COLUMN IF EXISTS is_verified",
    "ALTER TABLE users DROP COLUMN IF EXISTS phone",
    "ALTER TABLE users DROP COLUMN IF EXISTS full_name",
    "ALTER TABLE posts DROP COLUMN IF EXISTS edited_at",
    "ALTER TABLE posts DROP COLUMN IF EXISTS is_pinned",
    "DROP TABLE IF EXISTS user_settings",
    "DROP TABLE IF EXISTS schema_migrations",
    "DROP INDEX IF EXISTS idx_users_created",
]

for sql in cleanup_sqls:
    execute(sql)

print("🧹 Cleaned up — schema restored to original state.")

## 📚 Summary

### Key Takeaways

1. **Safe changes**: add nullable columns, add defaults, add indexes, add tables
2. **Dangerous changes**: drop columns, rename columns, narrow types, add NOT NULL
3. **3-step NOT NULL**: add nullable → backfill → set NOT NULL
4. **4-step rename**: add new → dual-write → migrate → drop old
5. **Track migrations**: use a `schema_migrations` table or a migration tool
6. **Use CONCURRENTLY**: for index creation on large production tables

### Interview Tip

> When discussing schema changes in an interview, mention:
> *"I'd do a zero-downtime migration — add the new column as nullable first,  
> backfill existing data, then add the NOT NULL constraint. This avoids  
> locking the table or breaking existing queries."*

### Series Complete! 🎉

You've now covered the full spectrum of data modeling:
1. **Relational basics** — entities, keys, relationships, indexes
2. **Denormalization** — when to trade consistency for read performance
3. **NoSQL models** — document, key-value, and wide-column alternatives
4. **Schema evolution** — changing your schema safely over time